# 02 - Exploratory Data Analysis (EDA)
Analisis mendalam terhadap dataset GDP Growth Indonesia.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('../data/eda_outputs', exist_ok=True)

df = pd.read_csv('../data/processed/dataset_indonesia.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# 1. Tren GDP Growth
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['Year'], df['GDP_Growth'], marker='o', color='#3B5BA5', linewidth=2)
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.fill_between(df['Year'], df['GDP_Growth'], 0,
                where=(df['GDP_Growth'] >= 0), alpha=0.2, color='green', label='Positif')
ax.fill_between(df['Year'], df['GDP_Growth'], 0,
                where=(df['GDP_Growth'] < 0), alpha=0.2, color='red', label='Negatif')
ax.set_title('Tren GDP Growth Indonesia (1991-2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Tahun')
ax.set_ylabel('GDP Growth (%)')
ax.legend()
# Annotasi krisis
ax.annotate('Krisis 1998', xy=(1998, df.loc[df['Year']==1998,'GDP_Growth'].values[0]),
            xytext=(2001, -10), arrowprops=dict(arrowstyle='->', color='red'), color='red')
ax.annotate('COVID-19 2020', xy=(2020, df.loc[df['Year']==2020,'GDP_Growth'].values[0]),
            xytext=(2016, -6), arrowprops=dict(arrowstyle='->', color='red'), color='red')
plt.tight_layout()
plt.savefig('../data/eda_outputs/gdp_trend.png', dpi=150)
plt.show()

In [ ]:
# 2. Heatmap Korelasi
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.drop('Year', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Heatmap Korelasi Antar Variabel', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eda_outputs/correlation_heatmap.png', dpi=150)
plt.show()
print('Korelasi dengan GDP_Growth:')
corr['GDP_Growth'].drop('GDP_Growth').sort_values(ascending=False)

In [ ]:
# 3. Scatter plots tiap variabel vs GDP Growth
features = ['Inflation', 'Unemployment', 'Population_Growth',
            'Exports', 'Imports', 'FDI', 'Exchange_Rate']
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
for i, feat in enumerate(features):
    axes[i].scatter(df[feat], df['GDP_Growth'], alpha=0.7, color='#3B5BA5', edgecolors='white', s=60)
    z = np.polyfit(df[feat].dropna(), df.loc[df[feat].notna(), 'GDP_Growth'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    axes[i].plot(x_line, p(x_line), 'r--', alpha=0.7)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('GDP Growth (%)')
    axes[i].set_title(f'{feat} vs GDP Growth')
axes[-1].set_visible(False)
plt.suptitle('Scatter Plot: Fitur vs GDP Growth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eda_outputs/scatter_plots.png', dpi=150)
plt.show()

In [ ]:
# 4. Distribusi semua variabel
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
all_cols = ['GDP_Growth'] + features
for i, col in enumerate(all_cols):
    axes[i].hist(df[col].dropna(), bins=15, color='#3B5BA5', edgecolor='white', alpha=0.8)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)
plt.suptitle('Distribusi Semua Variabel', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/eda_outputs/distributions.png', dpi=150)
plt.show()

In [ ]:
# 5. Statistik deskriptif lengkap
desc = df.describe().round(3)
desc.to_csv('../data/eda_outputs/descriptive_stats.csv')
desc